<a href="https://colab.research.google.com/github/sarahliu-mma/Quantitative-Trading-Pipeline_FIAM-Hackathon-2026/blob/main/notebooks/03b_data_preparation_model_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
import numpy as np
import pandas as pd

In [76]:
udt = pd.read_parquet('/content/drive/MyDrive/FIAM Hackathon/universe_panel.parquet')

In [77]:
print(udt.columns.tolist())

['permno', 'gvkey', 'exch_main', 'common', 'primary_sec', 'bidask', 'crsp_nyse', 'date', 'eom', 'me', 'prc', 'prc_local', 'dolvol', 'ret_exc', 'ret_exc_lead1m', 'obs_main', 'ff49', 'size_grp', 'ret_exc_wins', 'market_equity', 'div12m_me', 'eqnpo_12m', 'chcsho_12m', 'ret_1_0', 'ret_3_1', 'ret_6_1', 'ret_9_1', 'ret_12_1', 'ret_12_7', 'ret_60_12', 'seas_1_1an', 'seas_1_1na', 'seas_2_5an', 'seas_2_5na', 'at_gr1', 'sale_gr1', 'debt_gr3', 'sale_gr3', 'inv_gr1a', 'lti_gr1a', 'coa_gr1a', 'col_gr1a', 'cowc_gr1a', 'ncoa_gr1a', 'ncol_gr1a', 'nncoa_gr1a', 'noa_gr1a', 'fnl_gr1a', 'nfna_gr1a', 'tax_gr1a', 'ebit_sale', 'gp_at', 'cop_at', 'ope_be', 'ni_be', 'ebit_bev', 'netis_at', 'eqnetis_at', 'dbnetis_at', 'oaccruals_at', 'oaccruals_ni', 'taccruals_at', 'taccruals_ni', 'noa_at', 'opex_at', 'at_turnover', 'sale_bev', 'be_me', 'at_me', 'ni_me', 'sale_me', 'ocf_me', 'fcf_me', 'eqpo_me', 'eqnpo_me', 'bev_mev', 'ebitda_mev', 'ni_inc8q', 'ppeinv_gr1a', 'lnoa_gr1a', 'capx_gr1', 'capx_gr2', 'capx_gr3', 'sti

In [78]:
same_as_cur = np.isclose(udt['ret_exc_wins'],udt['ret_exc']).mean()
same_as_lead = np.isclose(udt['ret_exc_wins'], udt['ret_exc_lead1m']).mean()
print(same_as_cur,same_as_lead)

0.9999847458273482 5.084724217270266e-06


In [79]:
f = pd.read_csv('/content/drive/MyDrive/FIAM Hackathon/factor_char_list.csv')
jkp = f['variable'].tolist()

In [80]:
features_udt = [f for f in udt.columns if f in jkp]

In [81]:
features_udt.remove('prc')

In [82]:
identifier = ['permno','eom']
target = ['ret_exc_lead1m']

In [83]:
model_table = udt[identifier + features_udt + target].copy()

In [84]:
norm_modeltb = model_table.copy().reset_index(drop= True)

In [85]:
norm_modeltb[features_udt] = norm_modeltb.groupby('eom')[features_udt].rank(pct = True)

In [86]:
norm_modeltb.head()

,permno,eom,market_equity,div12m_me,eqnpo_12m,chcsho_12m,ret_1_0,ret_3_1,ret_6_1,ret_9_1,...,prc_highprc_252d,zero_trades_252d,betabab_1260d,rmax5_rvol_21d,age,qmj_prof,qmj_growth,qmj_safety,qmj,ret_exc_lead1m
0,10001,2015-01-31,0.067669,0.970242,0.807578,0.461796,0.175745,0.250261,0.133875,0.408303,...,0.256338,0.509859,0.051016,0.166725,0.696856,0.200617,0.130526,0.284297,0.055579,0.035918
1,10025,2015-01-31,0.191729,0.238060,0.957281,0.020499,0.121274,0.941953,0.923349,0.952708,...,0.472887,0.591901,0.589382,0.182940,0.718900,0.236283,0.009263,0.135819,0.021895,0.010997
2,10026,2015-01-31,0.604921,0.638501,0.573180,0.343645,0.235355,0.621828,0.769693,0.667148,...,0.451408,0.785915,0.327250,0.726119,0.718900,0.847737,0.154947,0.951078,0.778105,0.031309
3,10032,2015-01-31,0.527341,0.238060,0.556092,0.250839,0.310723,0.313173,0.387496,0.295668,...,0.434155,0.610211,0.596018,0.094466,0.696856,0.173525,0.312842,0.648649,0.227368,0.062307
4,10044,2015-01-31,0.045113,0.894563,0.791605,0.303764,0.828366,0.748001,0.375132,0.575090,...,0.875704,0.902465,0.051846,0.626718,0.718900,0.965021,0.131368,0.704755,0.673684,0.129391


In [87]:
print(norm_modeltb.shape, model_table.shape)
print(norm_modeltb.duplicated(subset=['permno', 'eom']).sum())
print(norm_modeltb[features_udt].min().min(), norm_modeltb[features_udt].max().max())
print(norm_modeltb['ret_exc_lead1m'].isna().sum())

(393335, 146) (393335, 146)
0
0.0002963841138114997 1.0
4132


In [88]:
norm_modeltb[norm_modeltb['ret_exc_lead1m'].isna()]['eom'].value_counts().sort_index().tail()

,count
eom,
2026-04-30,8
2026-05-31,13
2026-06-30,11
2026-07-31,11
2026-08-31,2510


In [89]:
norm_modeltb.to_parquet('/content/drive/MyDrive/FIAM Hackathon/model_table.parquet', index=False)